### NOTEBOOK 1.0 — Ingestão e Processamento dos Dados SICOR
### Autor: Pedro Boaventura
### Objetivo: Consolidar microdados BCB em base analítica Parquet

In [10]:
# =============================================================================
# NOTEBOOK 1.0 — Ingestão e Processamento dos Dados SICOR
# Autor: Pedro Boaventura
# Objetivo: Consolidar microdados BCB em base analítica Parquet
# =============================================================================

# %% [markdown]
# # 1. Setup e Dependências
#!pip install duckdb polars pyarrow pandas openpyxl

# %%
import duckdb
import polars as pl
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# Paths do projeto
ROOT       = Path("..") 
RAW        = ROOT / "data" / "raw"
DOMINIOS   = RAW / "tabelas_dominio"
INTERIM    = ROOT / "data" / "interim"
PROCESSED  = ROOT / "data" / "processed"

INTERIM.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

# Instância DuckDB persistente (arquivo no disco para cache entre sessões)
con = duckdb.connect(str(ROOT / "data" / "interim" / "sicor_db.duckdb"))
print("DuckDB version:", duckdb.__version__)
print("Polars version:", pl.__version__)

DuckDB version: 1.5.1
Polars version: 1.39.3


In [12]:
# %% [markdown]
# # 2. Ingestão das Tabelas de Domínio (pequenas, pandas OK)
# Estas tabelas são usadas como dicionários para enriquecer os microdados.

# %%
# Redefine load_dominio com remoção de BOM nos nomes de colunas
import csv

def load_dominio(filename: str, sep: str = ";", encoding: str = "latin1",
                 force_manual_split: bool = False) -> pd.DataFrame:
    """
    Carrega tabela de domínio do SICOR.
    
    force_manual_split=True: para arquivos como SituacaoOperacao.csv onde
    cada linha está inteiramente envolvida em aspas externas com aspas 
    internas duplicadas — padrão que o pandas read_csv não parseia corretamente.
    """
    filepath = DOMINIOS / filename

    if force_manual_split:
        rows = []
        with open(filepath, encoding=encoding) as f:
            for line in f:
                line = line.strip().strip('"')           # remove aspas externas
                parts = line.split(sep, maxsplit=1)      # split no 1º separador
                parts = [p.strip().strip('"') for p in parts]  # limpa aspas internas
                rows.append(parts)
        df = pd.DataFrame(rows[1:], columns=[c.lstrip('#').strip().upper()
                                             for c in rows[0]])
    else:
        df = pd.read_csv(filepath, sep=sep, encoding=encoding,
                         dtype=str, quotechar='"')
        df.columns = (df.columns
                      .str.lstrip('#')
                      .str.replace('\ufeff', '', regex=False)
                      .str.strip()
                      .str.upper())
    return df

# Carregar todas as tabelas de domínio
df_empreendimento   = load_dominio("Empreendimento.csv")
df_finalidade       = load_dominio("Finalidade.csv")
df_produto          = load_dominio("Produto.csv")
df_atividade        = load_dominio("Atividade.csv")
df_programa         = load_dominio("Programa.csv")
df_cat_emitente     = load_dominio("CategoriaEmitente.csv")
df_situacao_op      = load_dominio("SituacaoOperacao.csv", sep=",", force_manual_split=True)
df_fonte_recurso    = load_dominio("FonteRecursos.csv")
df_ifs              = load_dominio("SICOR_LISTA_IFS.csv")       # CNPJ_IF, NOME_IF, SEGMENTO_IF
df_municipios       = load_dominio("MunicipiosSicor.csv") # CD_IBGE_MUNICIPIO, lat, lon, UF

# Criar flag de cooperativa a partir do campo SEGMENTO_IF
df_ifs["IS_COOPERATIVA"] = df_ifs["SEGMENTO_IF"].str.contains(
    "COOPERATIVA|SICOOB|SICREDI|CRESOL", case=False, na=False
).astype(int)

# Identificar Sicoob especificamente
df_ifs["IS_SICOOB"] = df_ifs["NOME_IF"].str.contains("SICOOB", case=False, na=False).astype(int)

# Preview
print(f"IFs únicas: {len(df_ifs)}")
print(f"  - Cooperativas: {df_ifs['IS_COOPERATIVA'].sum()}")
print(f"  - Sicoob:       {df_ifs['IS_SICOOB'].sum()}")
df_ifs[df_ifs["IS_SICOOB"] == 1].head()
df_situacao_op.head(3)

IFs únicas: 644
  - Cooperativas: 561
  - Sicoob:       110


,CODIGO,DESCRICAO
0,1,Curso normal
1,2,Em atraso
2,3,Prorrogada


In [ ]:
# %% [markdown]
# # 3. Ingestão dos Microdados de Operações (GRANDE — via DuckDB)
# O DuckDB lê .gz diretamente sem extrair para disco!
# sicor_operacao_basica_estado: ~40 colunas, potencialmente 10M+ linhas por ano

# %%
# Listar arquivos disponíveis (busca case-insensitive: o BCB publica os nomes
# em maiúsculas e o glob é sensível a maiúsculas em Linux/macOS)
def listar_gz(prefixo: str) -> list[Path]:
    return sorted(f for f in RAW.glob("*.gz") if f.name.lower().startswith(prefixo))

operacao_files = listar_gz("sicor_operacao_basica_estado_")
saldos_files   = listar_gz("sicor_saldos_")

assert operacao_files and saldos_files, (
    "Arquivos .gz não encontrados em data/raw — veja a seção 'Dados' do README."
)

print("Arquivos de operações encontrados:")
for f in operacao_files:
    size_mb = f.stat().st_size / (1024**2)
    print(f"  {f.name:55s} {size_mb:8.1f} MB")

print("\nArquivos de saldos encontrados:")
for f in saldos_files:
    size_mb = f.stat().st_size / (1024**2)
    print(f"  {f.name:55s} {size_mb:8.1f} MB")

In [14]:
# %%
# -------------------------------------------------------------------
# 3.1 Criar view DuckDB para operações (todos os anos em uma só view)
# DuckDB suporta leitura de lista de arquivos .gz nativamente
# -------------------------------------------------------------------
# %%
# -------------------------------------------------------------------
# corrigindo BOM no header
# -------------------------------------------------------------------
operacao_paths = [str(f) for f in operacao_files]

con.execute(f"""
    CREATE OR REPLACE VIEW vw_operacoes_raw AS
    SELECT
        "#REF_BACEN" AS REF_BACEN,   -- Remove o BOM do nome da coluna
        * EXCLUDE ("#REF_BACEN")      -- Mantém todas as outras sem alteração
    FROM read_csv(
        {operacao_paths},
        sep=';',
        decimal_separator='.',
        header=true,
        ignore_errors=true,
        compression='gzip',
        all_varchar=false,
        parallel=true
    )
""")

# Validar que a coluna agora está limpa
schema_op = con.execute("DESCRIBE vw_operacoes_raw").df()
assert "REF_BACEN" in schema_op["column_name"].values, "❌ Coluna REF_BACEN não encontrada!"
print("✅ View criada com schema corrigido")
print(schema_op[["column_name", "column_type"]].to_string(index=False))

✅ View criada com schema corrigido
              column_name column_type
                REF_BACEN      BIGINT
                 NU_ORDEM      BIGINT
                  CNPJ_IF     VARCHAR
               DT_EMISSAO        DATE
            DT_VENCIMENTO        DATE
          CD_INST_CREDITO      BIGINT
        CD_CATEG_EMITENTE      BIGINT
         CD_FONTE_RECURSO     VARCHAR
       CNPJ_AGENTE_INVEST     VARCHAR
                CD_ESTADO     VARCHAR
CD_REF_BACEN_INVESTIMENTO     VARCHAR
           CD_TIPO_SEGURO      BIGINT
        CD_EMPREENDIMENTO      BIGINT
              CD_PROGRAMA     VARCHAR
    CD_TIPO_ENCARG_FINANC      BIGINT
        CD_TIPO_IRRIGACAO     VARCHAR
      CD_TIPO_AGRICULTURA      BIGINT
   CD_FASE_CICLO_PRODUCAO     VARCHAR
          CD_TIPO_CULTIVO     VARCHAR
     CD_TIPO_INTGR_CONSOR      BIGINT
     CD_TIPO_GRAO_SEMENTE      BIGINT
          VL_ALIQ_PROAGRO      DOUBLE
                 VL_JUROS      DOUBLE
VL_PRESTACAO_INVESTIMENTO      DOUBLE
             VL

In [15]:
# %%
# -------------------------------------------------------------------
# 3.2 Criar view para saldos
# sicor_saldos: REF_BACEN, NU_ORDEM, ANO_BASE, MES_BASE,
#               CD_SITUACAO_OPERACAO, VL_MEDIO_DIARIO,
#               VL_ULTIMO_DIA, VL_MEDIO_DIARIO_VINCENDO
# -------------------------------------------------------------------
# %%
# -------------------------------------------------------------------
# 3.2 Criar view de saldos — mesma correção de BOM
# -------------------------------------------------------------------
saldos_paths = [str(f) for f in saldos_files]

con.execute(f"""
    CREATE OR REPLACE VIEW vw_saldos_raw AS
    SELECT
        "#REF_BACEN" AS REF_BACEN,
        * EXCLUDE ("#REF_BACEN")
    FROM read_csv(
        {saldos_paths},
        sep=';',
        decimal_separator='.',
        header=true,
        ignore_errors=true,
        compression='gzip',
        all_varchar=false,
        parallel=true
    )
""")

schema_sd = con.execute("DESCRIBE vw_saldos_raw").df()
bom_cols = schema_sd[schema_sd["column_name"].str.startswith("#")]
if len(bom_cols) > 0:
    print("⚠️  Ainda há colunas com BOM:", bom_cols["column_name"].tolist())
else:
    print("✅ Schema de saldos OK — nenhum BOM remanescente")
    print(schema_sd[["column_name", "column_type"]].to_string(index=False))

✅ Schema de saldos OK — nenhum BOM remanescente
             column_name column_type
               REF_BACEN      BIGINT
                NU_ORDEM      BIGINT
                ANO_BASE      BIGINT
                MES_BASE      BIGINT
VL_MEDIO_DIARIO_VINCENDO      DOUBLE
           VL_ULTIMO_DIA      DOUBLE
         VL_MEDIO_DIARIO      DOUBLE
    CD_SITUACAO_OPERACAO      BIGINT


In [16]:
# %%
# Confirmar valores únicos de CD_ESTADO antes de filtrar
estados_disponiveis = con.execute("""
    SELECT CD_ESTADO, COUNT(*) AS qtd
    FROM vw_operacoes_raw
    GROUP BY CD_ESTADO
    ORDER BY qtd DESC
""").df()
print(estados_disponiveis)
# Se aparecer UF como "MT", "GO" etc., o filtro original está correto.
# Se aparecer código IBGE numérico, ajustar ESTADOS_FOCO para os códigos.

   CD_ESTADO     qtd
0         RS  883173
1         BA  856275
2         MG  789117
3         PR  558089
4         CE  448665
5         PI  429456
6         SC  417471
7         MA  375045
8         PE  367942
9         PB  294422
10        RN  193554
11        GO  176975
12        SP  153851
13        AL  149594
14        MT  126898
15        SE  126876
16        ES  126625
17        PA  108580
18        RO  106647
19        MS  102632
20        TO   47178
21        AC   22317
22        RJ   17114
23        AM   10928
24        RR    7705
25        AP    4309
26        DF    1184


In [17]:
# %% [markdown]
# # 4. Colunas Selecionadas e Filtros
# Foco no Centro-Oeste + Sul + Sudeste para o case Sicoob.
# Mantemos apenas colunas relevantes para análise de risco/oportunidade.
# Isso reduz 80%+ do volume antes de salvar em Parquet.

# %%
ESTADOS_FOCO = ["MT", "MS", "GO", "DF", "MG", "PR", "SP", "RS", "SC"]

# -------------------------------------------------------------------
# 4.1 Materializar operações filtradas e enriquecidas em Parquet
# -------------------------------------------------------------------
con.execute(f"""
COPY (
    SELECT
        op.REF_BACEN,
        op.NU_ORDEM,
        op.CNPJ_IF,
        op.DT_EMISSAO,
        op.DT_VENCIMENTO,
        op.CD_ESTADO,
        op.CD_EMPREENDIMENTO,
        op.CD_PROGRAMA,
        op.CD_FONTE_RECURSO,
        op.CD_CATEG_EMITENTE,
        op.CD_TIPO_AGRICULTURA,
        op.CD_TIPO_CULTIVO,
        op.VL_PARC_CREDITO,
        op.VL_JUROS,
        op.VL_AREA_FINANC,
        op.VL_RECEITA_BRUTA_ESPERADA,
        op.VL_PREV_PROD,
        op.VL_REC_PROPRIO,
        op.VL_PERC_CUSTO_EFET_TOTAL,
        -- Derivar prazo em meses
        DATEDIFF('month', op.DT_EMISSAO, op.DT_VENCIMENTO) AS PRAZO_MESES,
        -- Ano e mês de emissão para séries temporais
        YEAR(op.DT_EMISSAO)  AS ANO_EMISSAO,
        MONTH(op.DT_EMISSAO) AS MES_EMISSAO,
    FROM vw_operacoes_raw op
    WHERE op.CD_ESTADO IN ({", ".join(f"'{e}'" for e in ESTADOS_FOCO)})
) TO '{str(INTERIM / "operacoes.parquet")}'
(FORMAT PARQUET, COMPRESSION SNAPPY)
""")

size = (INTERIM / "operacoes.parquet").stat().st_size / (1024**2)
n    = con.execute(f"SELECT COUNT(*) FROM '{str(INTERIM / 'operacoes.parquet')}'").fetchone()[0]
print(f"✅ Parquet salvo: {n:,} registros | {size:.1f} MB")

✅ Parquet salvo: 3,209,390 registros | 88.2 MB


In [18]:
# %%
# -------------------------------------------------------------------
# 4.2 Materializar saldos filtrados (apenas REF_BACEN que existem em operações)
# -------------------------------------------------------------------
con.execute(f"""
COPY (
    SELECT
        s.REF_BACEN,
        s.NU_ORDEM,
        s.ANO_BASE,
        s.MES_BASE,
        s.CD_SITUACAO_OPERACAO,
        s.VL_MEDIO_DIARIO,
        s.VL_ULTIMO_DIA,
        s.VL_MEDIO_DIARIO_VINCENDO,
        -- VL_VENCIDO calculado: saldo total menos parcela ainda a vencer
        GREATEST(s.VL_ULTIMO_DIA - s.VL_MEDIO_DIARIO_VINCENDO, 0) AS VL_VENCIDO
    FROM vw_saldos_raw s
    SEMI JOIN (
        SELECT DISTINCT REF_BACEN FROM '{str(INTERIM / "operacoes.parquet")}'
    ) op ON s.REF_BACEN = op.REF_BACEN
) TO '{str(INTERIM / "saldos.parquet")}'
(FORMAT PARQUET, COMPRESSION SNAPPY)
""")

size = (INTERIM / "saldos.parquet").stat().st_size / (1024**2)
n    = con.execute(f"SELECT COUNT(*) FROM '{str(INTERIM / 'saldos.parquet')}'").fetchone()[0]
print(f"✅ Parquet de saldos: {n:,} registros | {size:.1f} MB")

✅ Parquet de saldos: 39,417,043 registros | 1056.8 MB


In [ ]:
# %%
# Inspecionar os códigos reais de situação antes de definir flags
df_sit_check = con.execute(f"""
    SELECT CD_SITUACAO_OPERACAO, COUNT(*) AS qtd
    FROM '{str(INTERIM / "saldos.parquet")}'
    GROUP BY 1 ORDER BY 2 DESC
""").df()

# Cruzar com tabela de domínio para mapear os códigos
print(df_situacao_op.to_string(index=False))
print("\nDistribuição nos saldos:")
print(df_sit_check.to_string(index=False))

In [20]:
# %% [markdown]
# # 5. Registro das Tabelas de Domínio no DuckDB
# Registrar como tabelas in-memory para facilitar os JOINs seguintes

# %%
con.register("dom_ifs",        df_ifs)
con.register("dom_empreend",   df_empreendimento)
con.register("dom_situacao",   df_situacao_op)
con.register("dom_programa",   df_programa)
con.register("dom_fonte",      df_fonte_recurso)
con.register("dom_municipios", df_municipios)
con.register("dom_cat_emit",   df_cat_emitente)

print("✅ Domínios registrados no DuckDB")

✅ Domínios registrados no DuckDB


In [21]:
# %%
# Verificar colunas reais do Empreendimento para usar os nomes corretos no JOIN
print("Colunas de Empreendimento:", df_empreendimento.columns.tolist())
print(df_empreendimento.head(3).to_string())
print("\nColunas de Programa:", df_programa.columns.tolist())
print("\nColunas de FonteRecursos:", df_fonte_recurso.columns.tolist())
print("\nColunas de CategoriaEmitente:", df_cat_emitente.columns.tolist())
print("\nColunas de SituacaoOperacao:", df_situacao_op.columns.tolist())
print(df_situacao_op.head(3).to_string())

Colunas de Empreendimento: ['CODIGO', 'DATA_INICIO', 'DATA_FIM', 'FINALIDADE', 'ATIVIDADE', 'MODALIDADE', 'PRODUTO', 'VARIEDADE', 'CESTA', 'ZONEAMENTO', 'UNIDADE_MEDIDA', 'UNIDADE_MEDIDA_PREVISAO', 'CONSORCIO', 'CEDULA_MAE', 'CD_TIPO_CULTURA']
           CODIGO DATA_INICIO DATA_FIM FINALIDADE ATIVIDADE MODALIDADE                        PRODUTO      VARIEDADE                          CESTA   ZONEAMENTO UNIDADE_MEDIDA UNIDADE_MEDIDA_PREVISAO        CONSORCIO CEDULA_MAE CD_TIPO_CULTURA
0  12018888000002  07/01/2020      NaN    Custeio  Agrícola    LAVOURA  CITRONELA (CYMBOPOGON NARDUS)  NÃO SE APLICA  Ano Civil / Ano de Exploração  Não zoneado        Hectare                Tonelada  Não consorciado          0               2
1  12017340000043  16/03/2017      NaN    Custeio  Agrícola    LAVOURA                           VIME  NÃO SE APLICA    Fruticultura / Olericultura          NaN        Hectare                Tonelada  Não consorciado          0               2
2  12017340000042  01/01

In [22]:
# %%
# Ajustar os nomes de coluna abaixo conforme o output da célula de inspeção acima
# Exemplos típicos do BCB — confirme antes de rodar:
#   Empreendimento: CD_EMPREENDIMENTO, MODALIDADE, FINALIDADE, PRODUTO, ATIVIDADE
#   Programa:       CD_PROGRAMA, DESCRICAO
#   FonteRecursos:  CD_FONTE_RECURSO, DESCRICAO
#   CategoriaEmit:  CD_CATEG_EMITENTE, DESCRICAO
#   SituacaoOp:     CODIGO, DESCRICAO  (após remoção do BOM)

con.execute(f"""
COPY (
    SELECT
        -- Chaves
        op.REF_BACEN,
        op.NU_ORDEM,

        -- Datas e prazo
        op.DT_EMISSAO,
        op.DT_VENCIMENTO,
        op.ANO_EMISSAO,
        op.MES_EMISSAO,
        op.PRAZO_MESES,

        -- Localização
        op.CD_ESTADO,

        -- Identificação da IF
        op.CNPJ_IF,
        if_.NOME_IF,
        if_.SEGMENTO_IF,
        if_.IS_COOPERATIVA,
        if_.IS_SICOOB,

        -- Empreendimento (produto/modalidade/finalidade)
        op.CD_EMPREENDIMENTO,
        empr.MODALIDADE,
        empr.FINALIDADE,
        empr.PRODUTO,
        empr.ATIVIDADE,

        -- Programa e fonte de recursos
        op.CD_PROGRAMA,
        prog.DESCRICAO          AS DESC_PROGRAMA,
        op.CD_FONTE_RECURSO,
        fonte.DESCRICAO         AS DESC_FONTE_RECURSO,

        -- Tipo de agricultura e cultivo
        op.CD_TIPO_AGRICULTURA,
        op.CD_TIPO_CULTIVO,

        -- Categoria do emitente (Agricultor Familiar, Médio, Grande)
        op.CD_CATEG_EMITENTE,
        cat.DESCRICAO           AS DESC_CATEG_EMITENTE,

        -- Valores financeiros
        op.VL_PARC_CREDITO,
        op.VL_JUROS,
        op.VL_AREA_FINANC,
        op.VL_RECEITA_BRUTA_ESPERADA,
        op.VL_PREV_PROD,
        op.VL_REC_PROPRIO,
        op.VL_PERC_CUSTO_EFET_TOTAL,

        -- Feature: LTV (proxy Loan-to-Value)
        CASE
            WHEN op.VL_RECEITA_BRUTA_ESPERADA > 0
            THEN op.VL_PARC_CREDITO / op.VL_RECEITA_BRUTA_ESPERADA
            ELSE NULL
        END AS LTV,

        -- Saldo mais recente (snapshot do último mês disponível)
        sal.ANO_BASE,
        sal.MES_BASE,
        sal.CD_SITUACAO_OPERACAO,
        sit.DESCRICAO           AS DESC_SITUACAO_OPERACAO,
        sal.VL_MEDIO_DIARIO,
        sal.VL_ULTIMO_DIA,
        sal.VL_MEDIO_DIARIO_VINCENDO,
        sal.VL_VENCIDO,

        -- Nível de risco categórico (coluna analítica principal)
        CASE
            WHEN sal.CD_SITUACAO_OPERACAO IN (8, 9, 11) THEN 'INADIMPLENTE_GRAVE'
            WHEN sal.CD_SITUACAO_OPERACAO IN (2, 12, 13) THEN 'INADIMPLENTE'
            WHEN sal.CD_SITUACAO_OPERACAO IN (4, 5, 6)   THEN 'RENEGOCIADA'
            WHEN sal.CD_SITUACAO_OPERACAO = 3             THEN 'PRORROGADA'
            WHEN sal.CD_SITUACAO_OPERACAO = 7             THEN 'LIQUIDADA'
            WHEN sal.CD_SITUACAO_OPERACAO = 1             THEN 'NORMAL'
            ELSE 'DESCONHECIDO'
        END AS NIVEL_RISCO,

        -- Flag binária estrita (KPI oficial BACEN)
        CASE
            WHEN sal.CD_SITUACAO_OPERACAO IN (2, 8, 9, 11, 12, 13) THEN 1
            ELSE 0
        END AS FLAG_INADIMPLENTE,

        -- Flag risco ampliado (inclui renegociadas — carteira sob pressão)
        CASE
            WHEN sal.CD_SITUACAO_OPERACAO IN (2, 4, 5, 6, 8, 9, 11, 12, 13) THEN 1
            ELSE 0
        END AS FLAG_RISCO_AMPLIADO,

        -- Flag operação ativa (denominador correto para taxas)
        CASE
            WHEN sal.CD_SITUACAO_OPERACAO NOT IN (7, 10) THEN 1
            ELSE 0
        END AS FLAG_ATIVA

    FROM '{str(INTERIM / "operacoes.parquet")}' op

    LEFT JOIN dom_ifs      if_   ON op.CNPJ_IF          = if_.CNPJ_IF
    LEFT JOIN dom_empreend empr  ON op.CD_EMPREENDIMENTO = CAST(empr.CODIGO AS BIGINT)
    LEFT JOIN dom_programa prog  ON op.CD_PROGRAMA       = prog.CODIGO
    LEFT JOIN dom_fonte    fonte ON op.CD_FONTE_RECURSO  = fonte.CODIGO
    LEFT JOIN dom_cat_emit cat   ON op.CD_CATEG_EMITENTE = CAST(cat.CODIGO AS BIGINT)

    -- Saldo mais recente por operação (QUALIFY = ROW_NUMBER inline)
    LEFT JOIN (
        SELECT *
        FROM '{str(INTERIM / "saldos.parquet")}'
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY REF_BACEN, NU_ORDEM
            ORDER BY ANO_BASE DESC, MES_BASE DESC
        ) = 1
    ) sal ON op.REF_BACEN = sal.REF_BACEN AND op.NU_ORDEM = sal.NU_ORDEM

    -- JOIN situação: CD_SITUACAO_OPERACAO é BIGINT, CODIGO no domínio é VARCHAR após load
    LEFT JOIN dom_situacao sit ON sal.CD_SITUACAO_OPERACAO = CAST(sit.CODIGO AS BIGINT)

) TO '{str(PROCESSED / "base_analitica.parquet")}'
(FORMAT PARQUET, COMPRESSION SNAPPY, ROW_GROUP_SIZE 100000)
""")

size = (PROCESSED / "base_analitica.parquet").stat().st_size / (1024**2)
n    = con.execute(f"SELECT COUNT(*) FROM '{str(PROCESSED / 'base_analitica.parquet')}'").fetchone()[0]
print(f"✅ Base analítica: {n:,} registros | {size:.1f} MB")

✅ Base analítica: 3,209,390 registros | 170.5 MB


In [23]:
# %% [markdown]
# # 7. Validação e Sanity Checks

# %%
df_check = con.execute(f"""
SELECT
    COUNT(*)                                         AS total_contratos,
    COUNT(DISTINCT REF_BACEN)                        AS operacoes_unicas,
    COUNT(DISTINCT CNPJ_IF)                          AS ifs_distintas,
    SUM(VL_PARC_CREDITO) / 1e9                       AS carteira_total_bilhoes,
    AVG(VL_PARC_CREDITO)                             AS ticket_medio,
    SUM(FLAG_INADIMPLENTE) * 100.0 / COUNT(*)        AS pct_inadimplente,
    COUNT(DISTINCT CD_ESTADO)                        AS estados_cobertos,
    MIN(ANO_EMISSAO)                                 AS ano_inicio,
    MAX(ANO_EMISSAO)                                 AS ano_fim
FROM '{str(PROCESSED / "base_analitica.parquet")}'
""").df()

print("=== RESUMO DA BASE ANALÍTICA ===")
print(df_check.T.to_string())

=== RESUMO DA BASE ANALÍTICA ===
                                   0
total_contratos         3.209390e+06
operacoes_unicas        2.965717e+06
ifs_distintas           4.290000e+02
carteira_total_bilhoes  9.008300e+02
ticket_medio            2.806857e+05
pct_inadimplente        1.839228e+00
estados_cobertos        9.000000e+00
ano_inicio              2.023000e+03
ano_fim                 2.025000e+03


In [24]:
# %% [markdown]
# # 8. Exportar Amostra para Desenvolvimento Rápido
# Para exploração no notebook 2.0 sem recarregar tudo

# %%
# Amostra estratificada por estado e ano (10% ou max 500k linhas)
con.execute(f"""
COPY (
    SELECT *
    FROM '{str(PROCESSED / "base_analitica.parquet")}'
    USING SAMPLE 10 PERCENT (bernoulli, 42)
) TO '{str(PROCESSED / "base_analitica_sample.parquet")}'
(FORMAT PARQUET, COMPRESSION SNAPPY)
""")

n_sample = con.execute(
    f"SELECT COUNT(*) FROM '{str(PROCESSED / 'base_analitica_sample.parquet')}'"
).fetchone()[0]
print(f"✅ Sample salvo: {n_sample:,} registros")

✅ Sample salvo: 321,453 registros


In [25]:
# %% [markdown]
# # 9. Série Temporal de Saldos (Série Mensal para KPIs)
# Gerar tabela agregada por ANO_BASE + MES_BASE para análises de tendência

# %%
df_serie_mensal = con.execute(f"""
    SELECT
        s.ANO_BASE,
        s.MES_BASE,
        STRFTIME(MAKE_DATE(s.ANO_BASE, s.MES_BASE, 1), '%Y-%m') AS ANO_MES,
        COUNT(DISTINCT s.REF_BACEN)               AS num_operacoes,
        SUM(s.VL_ULTIMO_DIA) / 1e9                AS saldo_total_bi,
        SUM(s.VL_VENCIDO) / 1e9                   AS saldo_vencido_bi,
        SUM(s.VL_VENCIDO) * 100.0
            / NULLIF(SUM(s.VL_ULTIMO_DIA), 0)     AS taxa_inadimplencia_pct,

        -- Recorte Sicoob
        SUM(CASE WHEN if_.IS_SICOOB = 1 THEN s.VL_ULTIMO_DIA ELSE 0 END) / 1e9
                                                  AS saldo_sicoob_bi,
        SUM(CASE WHEN if_.IS_SICOOB = 1 THEN s.VL_VENCIDO ELSE 0 END) * 100.0
            / NULLIF(SUM(CASE WHEN if_.IS_SICOOB = 1 THEN s.VL_ULTIMO_DIA ELSE 0 END), 0)
                                                  AS taxa_inadimplencia_sicoob_pct
    FROM '{str(INTERIM / "saldos.parquet")}' s
    JOIN '{str(PROCESSED / "base_analitica.parquet")}' op
        ON s.REF_BACEN = op.REF_BACEN AND s.NU_ORDEM = op.NU_ORDEM
    LEFT JOIN dom_ifs if_ ON op.CNPJ_IF = if_.CNPJ_IF
    GROUP BY s.ANO_BASE, s.MES_BASE
    ORDER BY s.ANO_BASE, s.MES_BASE
""").df()

df_serie_mensal.to_csv(str(PROCESSED / "serie_mensal_inadimplencia.csv"), index=False)
print("✅ Série mensal salva!")
print(df_serie_mensal.tail(12).to_string(index=False))

✅ Série mensal salva!
 ANO_BASE  MES_BASE ANO_MES  num_operacoes  saldo_total_bi  saldo_vencido_bi  taxa_inadimplencia_pct  saldo_sicoob_bi  taxa_inadimplencia_sicoob_pct
     2025         3 2025-03        1291487      420.597795         27.906218                6.634894        15.001158                       4.186719
     2025         4 2025-04        1303550      423.270329         22.813197                5.389746        15.214844                       4.438411
     2025         5 2025-05        1309526      434.837498         32.773393                7.536929        15.418411                       4.452627
     2025         6 2025-06        1306668      437.661558         28.020512                6.402324        16.245688                       6.103657
     2025         7 2025-07        1311362      436.575128         30.173368                6.911380        16.865858                       5.127961
     2025         8 2025-08        1347763      446.276285         34.920241        